# División del dataset V5

Crea un dataset independiente por producto, pero conserva todas las fechas. La columna objetivo se llama `demanda` y cambia según el plato. La división es cronológica para evitar fuga de información.

In [2]:
# ============================================================
# DIVISIÓN DEL DATASET POR TIPO DE PLATO - V5
# Enfoque: un dataset completo por producto con todas las fechas
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os
import json

# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

# Carpeta base donde está el archivo y donde se guardarán los resultados
BASE = "/content"

# Archivo de entrada
INPUT_PATH = f"{BASE}/dataset_preparado_v3.csv"

# Carpeta donde se crearán los datasets por producto
OUTPUT_DIR = f"{BASE}/datasets_por_plato"

# Archivo resumen de la división
OUTPUT_RESUMEN = f"{BASE}/resumen_division_dataset.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

PRODUCTOS = ["almuerzo", "sopa", "fanesca", "colada_morada"]

VARIABLES_PREDICTORAS = [
    "anio",
    "mes",
    "dia_mes",
    "dia_semana_num",
    "semana_anio",

    "es_fanesca_temporada",
    "es_colada_temporada",
    "es_inicio_mes",
    "es_quincena",
    "es_fin_mes",

    "es_lunes",
    "es_martes",
    "es_miercoles",
    "es_jueves",
    "es_viernes",

    "mes_sin",
    "mes_cos",
    "dia_semana_sin",
    "dia_semana_cos",

    "tendencia",
    "tendencia_log",
    "crecimiento_anual",

    "preciomenu",
    "preciosopa",
    "fanesca_precio",
    "coladamorada_precio"
]

TRAIN_SIZE = 0.70
VALIDACION_SIZE = 0.15
PRUEBA_SIZE = 0.15

# Para productos estacionales, además del split general 70/15/15,
# se crea una evaluación especial para revisar temporada real.
PRODUCTOS_ESTACIONALES = ["fanesca", "colada_morada"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# ============================================================
# 2. CARGAR DATASET PREPARADO
# ============================================================

from pathlib import Path

def leer_dataset_preparado(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Si es Excel
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    # Si es CSV, probamos diferentes separadores y codificaciones
    if path.suffix.lower() == ".csv":
        intentos = [
            {"sep": ";", "encoding": "utf-8-sig"},
            {"sep": ",", "encoding": "utf-8-sig"},
            {"sep": ";", "encoding": "latin-1"},
            {"sep": ",", "encoding": "latin-1"},
            {"sep": None, "encoding": "utf-8-sig"},
            {"sep": None, "encoding": "latin-1"},
        ]

        ultimo_error = None

        for intento in intentos:
            try:
                if intento["sep"] is None:
                    df_temp = pd.read_csv(
                        path,
                        sep=None,
                        engine="python",
                        encoding=intento["encoding"]
                    )
                else:
                    df_temp = pd.read_csv(
                        path,
                        sep=intento["sep"],
                        encoding=intento["encoding"],
                        engine="python"
                    )

                # Validación mínima: debe tener varias columnas
                if df_temp.shape[1] > 1:
                    print(
                        f"Archivo leído correctamente con separador "
                        f"{repr(intento['sep'])} y encoding {intento['encoding']}"
                    )
                    return df_temp

            except Exception as e:
                ultimo_error = e

        raise ValueError(
            f"No se pudo leer el CSV correctamente. Último error: {ultimo_error}"
        )

    raise ValueError("Formato no soportado. Usa .xlsx, .xls o .csv")


df = leer_dataset_preparado(INPUT_PATH)

# Normalizar nombre de columnas por seguridad
df.columns = [str(c).strip() for c in df.columns]

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

faltantes = [
    c for c in ["fecha"] + VARIABLES_PREDICTORAS + PRODUCTOS
    if c not in df.columns
]

if faltantes:
    raise ValueError(f"Faltan columnas en dataset_preparado: {faltantes}")

print("Registros del dataset preparado:", len(df))
print("Rango:", df["fecha"].min().date(), "a", df["fecha"].max().date())

display(df.head())

Archivo leído correctamente con separador ';' y encoding utf-8-sig
Registros del dataset preparado: 740
Rango: 2023-01-02 a 2025-12-31


,fecha,anio,mes,dia_mes,dia_semana_num,semana_anio,es_fanesca_temporada,es_colada_temporada,es_inicio_mes,es_quincena,...,tendencia_log,crecimiento_anual,preciomenu,preciosopa,fanesca_precio,coladamorada_precio,almuerzo,sopa,fanesca,colada_morada
0,2023-01-02,2023,1,2,0,1,0,0,1,0,...,0,0,4,"1,5",7,2,104,72,0,0
1,2023-01-03,2023,1,3,1,1,0,0,1,0,...,"0,693147181",0,4,"1,5",7,2,106,54,0,0
2,2023-01-04,2023,1,4,2,1,0,0,1,0,...,"1,098612289",0,4,"1,5",7,2,97,72,0,0
3,2023-01-05,2023,1,5,3,1,0,0,1,0,...,"1,386294361",0,4,"1,5",7,2,103,0,0,0
4,2023-01-06,2023,1,6,4,1,0,0,0,0,...,"1,609437912",0,4,"1,5",7,2,123,0,0,0


In [8]:
# ============================================================
# 3. FUNCIÓN DE SPLIT CRONOLÓGICO
# ============================================================

def split_cronologico(df_producto, train_size=0.70, validacion_size=0.15):
    n = len(df_producto)

    n_train = int(n * train_size)
    n_val = int(n * validacion_size)

    train = df_producto.iloc[:n_train].copy()
    validacion = df_producto.iloc[n_train:n_train + n_val].copy()
    prueba = df_producto.iloc[n_train + n_val:].copy()

    return train, validacion, prueba



In [9]:
# ============================================================
# 4. CREAR DATASETS POR PRODUCTO
# ============================================================

resumen = []

for producto in PRODUCTOS:
    print(f"\nProcesando producto: {producto}")

    output_producto = f"{OUTPUT_DIR}/{producto}"
    os.makedirs(output_producto, exist_ok=True)

    # Importante:
    # Se conservan TODAS las fechas del dataset preparado.
    # Solo cambia la columna objetivo: demanda = producto.
    # Esto permite que el modelo aprenda tanto días con venta como días sin venta.
    columnas_producto = ["fecha"] + VARIABLES_PREDICTORAS + [producto]

    df_producto = df[columnas_producto].copy()
    df_producto = df_producto.rename(columns={producto: "demanda"})
    df_producto["demanda"] = pd.to_numeric(df_producto["demanda"], errors="coerce").fillna(0)
    df_producto = df_producto.sort_values("fecha").reset_index(drop=True)

    train, validacion, prueba = split_cronologico(
        df_producto,
        train_size=TRAIN_SIZE,
        validacion_size=VALIDACION_SIZE
    )

    df_producto.to_excel(f"{output_producto}/dataset_{producto}_completo.xlsx", index=False)
    train.to_excel(f"{output_producto}/train_{producto}.xlsx", index=False)
    validacion.to_excel(f"{output_producto}/validacion_{producto}.xlsx", index=False)
    prueba.to_excel(f"{output_producto}/prueba_{producto}.xlsx", index=False)

    # Evaluación especial para productos estacionales:
    # train_estacional = años anteriores al último año con demanda
    # prueba_estacional = último año con demanda en temporada
    estacional_creado = False
    if producto in PRODUCTOS_ESTACIONALES:
        df_con_demanda = df_producto[df_producto["demanda"] > 0].copy()

        if not df_con_demanda.empty:
            anios_demanda = sorted(df_con_demanda["fecha"].dt.year.unique().tolist())

            if len(anios_demanda) >= 2:
                anio_test = anios_demanda[-1]

                train_estacional = df_producto[
                    (df_producto["fecha"].dt.year < anio_test)
                ].copy()

                prueba_estacional = df_producto[
                    (df_producto["fecha"].dt.year == anio_test) &
                    (df_producto["demanda"].gt(0) | df_producto.get(f"es_{producto}_temporada", pd.Series(False, index=df_producto.index)).astype(bool))
                ].copy()

                # Para fanesca y colada, el nombre exacto de variable estacional es diferente.
                if producto == "fanesca" and "es_fanesca_temporada" in df_producto.columns:
                    prueba_estacional = df_producto[
                        (df_producto["fecha"].dt.year == anio_test) &
                        (df_producto["es_fanesca_temporada"] == 1)
                    ].copy()

                if producto == "colada_morada" and "es_colada_temporada" in df_producto.columns:
                    prueba_estacional = df_producto[
                        (df_producto["fecha"].dt.year == anio_test) &
                        (df_producto["es_colada_temporada"] == 1)
                    ].copy()

                if not train_estacional.empty and not prueba_estacional.empty:
                    train_estacional.to_excel(f"{output_producto}/train_estacional_{producto}.xlsx", index=False)
                    prueba_estacional.to_excel(f"{output_producto}/prueba_estacional_{producto}.xlsx", index=False)
                    estacional_creado = True

    resumen.append({
        "producto": producto,
        "filas_total": len(df_producto),
        "filas_train": len(train),
        "filas_validacion": len(validacion),
        "filas_prueba": len(prueba),
        "fecha_min_train": train["fecha"].min().date() if len(train) else None,
        "fecha_max_train": train["fecha"].max().date() if len(train) else None,
        "fecha_min_validacion": validacion["fecha"].min().date() if len(validacion) else None,
        "fecha_max_validacion": validacion["fecha"].max().date() if len(validacion) else None,
        "fecha_min_prueba": prueba["fecha"].min().date() if len(prueba) else None,
        "fecha_max_prueba": prueba["fecha"].max().date() if len(prueba) else None,
        "total_demanda": int(df_producto["demanda"].sum()),
        "dias_con_demanda": int((df_producto["demanda"] > 0).sum()),
        "evaluacion_estacional_creada": estacional_creado
    })




Procesando producto: almuerzo

Procesando producto: sopa

Procesando producto: fanesca

Procesando producto: colada_morada


In [10]:
# ============================================================
# 5. EXPORTAR RESUMEN
# ============================================================

df_resumen = pd.DataFrame(resumen)
df_resumen.to_excel(OUTPUT_RESUMEN, index=False)

print("\nDivisión finalizada correctamente.")
print("Carpeta de salida:", OUTPUT_DIR)
print("Resumen:", OUTPUT_RESUMEN)

display(df_resumen)


División finalizada correctamente.
Carpeta de salida: /content/datasets_por_plato
Resumen: /content/resumen_division_dataset.xlsx


,producto,filas_total,filas_train,filas_validacion,filas_prueba,fecha_min_train,fecha_max_train,fecha_min_validacion,fecha_max_validacion,fecha_min_prueba,fecha_max_prueba,total_demanda,dias_con_demanda,evaluacion_estacional_creada
0,almuerzo,740,518,111,111,2023-01-02,2025-02-06,2025-02-07,2025-07-17,2025-07-18,2025-12-31,75225,737,False
1,sopa,740,518,111,111,2023-01-02,2025-02-06,2025-02-07,2025-07-17,2025-07-18,2025-12-31,44370,661,False
2,fanesca,740,518,111,111,2023-01-02,2025-02-06,2025-02-07,2025-07-17,2025-07-18,2025-12-31,652,35,True
3,colada_morada,740,518,111,111,2023-01-02,2025-02-06,2025-02-07,2025-07-17,2025-07-18,2025-12-31,1567,63,True


In [11]:
import shutil
from google.colab import files

shutil.make_archive("/content/datasets_por_plato", "zip", "/content/datasets_por_plato")

files.download("/content/datasets_por_plato.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>